In [1]:
import os
import getpass
import mysql.connector

RDS_HOST = (
    "ecommerce-recommendation-db."
    "c9u0ucy2umao.ap-south-1.rds.amazonaws.com"
)
RDS_PORT = 3306
RDS_USER = "adminuser"
RDS_PASSWORD = getpass.getpass("Enter RDS password: ")

connection = mysql.connector.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    user=RDS_USER,
    password=RDS_PASSWORD,
    connection_timeout=20
)

cursor = connection.cursor()

cursor.execute(
    """
    CREATE DATABASE IF NOT EXISTS ecommerce_bigdata
    CHARACTER SET utf8mb4
    COLLATE utf8mb4_unicode_ci
    """
)

connection.commit()

print("Database ecommerce_bigdata is ready.")

cursor.close()
connection.close()

Enter RDS password:  ········


Database ecommerce_bigdata is ready.


In [6]:
import os
import getpass

os.environ["RDS_PASSWORD"] = getpass.getpass(
    "Enter the Amazon RDS password: "
)

Enter the Amazon RDS password:  ········


In [9]:
RDS_HOST = "ecommerce-recommendation-db.c9u0ucy2umao.ap-south-1.rds.amazonaws.com"
RDS_PORT = 3306
RDS_DATABASE = "ecommerce_analytics"
RDS_USER = "adminuser"
RDS_PASSWORD = os.environ["RDS_PASSWORD"]

In [10]:
import pymysql

conn = pymysql.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    user=RDS_USER,
    password=RDS_PASSWORD,
    database=RDS_DATABASE,
    charset="utf8mb4",
    cursorclass=pymysql.cursors.DictCursor,
    connect_timeout=20,
    autocommit=False,
    ssl_disabled=True
)

print("Connected successfully to Amazon RDS!")

Connected successfully to Amazon RDS!


In [14]:
import os

print(os.path.exists("global-bundle.pem"))

True


In [19]:
import json
import os
from pathlib import Path

import pymysql


# ==================================================
# AMAZON RDS CONFIGURATION
# ==================================================
RDS_HOST = (
    "ecommerce-recommendation-db."
    "c9u0ucy2umao.ap-south-1.rds.amazonaws.com"
)

RDS_PORT = 3306
RDS_DATABASE = "ecommerce_analytics"
RDS_USER = "adminuser"
RDS_PASSWORD = os.getenv("RDS_PASSWORD")

PRODUCTS_FILE = Path("products.json")
SSL_CERTIFICATE = Path("global-bundle.pem")

TABLE_NAME = "products_product"

# Change to webp or jpg when necessary.
IMAGE_EXTENSION = "png"


# ==================================================
# VALIDATION
# ==================================================
if not RDS_PASSWORD:
    raise ValueError(
        "RDS_PASSWORD is not set.\n"
        "Run this first:\n"
        'os.environ["RDS_PASSWORD"] = "your-password"'
    )

if not PRODUCTS_FILE.exists():
    raise FileNotFoundError(
        f"products.json was not found at:\n"
        f"{PRODUCTS_FILE.resolve()}"
    )

if not SSL_CERTIFICATE.exists():
    raise FileNotFoundError(
        f"global-bundle.pem was not found at:\n"
        f"{SSL_CERTIFICATE.resolve()}"
    )


# ==================================================
# LOAD PRODUCTS.JSON
# ==================================================
with PRODUCTS_FILE.open("r", encoding="utf-8") as file:
    json_data = json.load(file)


# Support both:
#
# [
#   {...},
#   {...}
# ]
#
# and:
#
# {
#   "products": [
#       {...},
#       {...}
#   ]
# }
if isinstance(json_data, list):
    products = json_data

elif (
    isinstance(json_data, dict)
    and isinstance(json_data.get("products"), list)
):
    products = json_data["products"]

else:
    raise ValueError(
        "products.json must contain either a JSON list "
        "or an object containing a 'products' list."
    )


print(f"Products found in products.json: {len(products)}")

if products:
    print("\nFields found in the first product:")
    print(sorted(products[0].keys()))


# ==================================================
# HELPER FUNCTIONS
# ==================================================
def clean_text(value, default=""):
    if value is None:
        return default

    return str(value).strip()


def clean_price(value):
    if value in (None, ""):
        return 0.0

    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0


def clean_stock(value):
    if value in (None, ""):
        return 0

    try:
        return int(value)
    except (TypeError, ValueError):
        return 0


def clean_boolean(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str):
        return value.strip().lower() in {
            "true",
            "1",
            "yes",
            "active",
        }

    return bool(value)


def create_description(product):
    """
    The current products_product table does not have separate
    columns for subcategory, stock, active status, price history,
    or creation date.

    Therefore, those values are safely included in description.
    """

    original_description = clean_text(
        product.get("description")
    )

    subcategory = clean_text(
        product.get("subcategory_id"),
        "Not specified",
    )

    stock = clean_stock(
        product.get("current_stock")
    )

    active = clean_boolean(
        product.get("is_active", True)
    )

    creation_date = clean_text(
        product.get("creation_date"),
        "Not specified",
    )

    parts = []

    if original_description:
        parts.append(original_description)

    parts.extend(
        [
            f"Subcategory: {subcategory}",
            f"Available stock: {stock}",
            f"Active: {'Yes' if active else 'No'}",
            f"Created: {creation_date}",
        ]
    )

    return " | ".join(parts)


def create_category(product):
    """
    Store the readable category when it exists.
    Otherwise store category_id.
    """

    return clean_text(
        product.get("category")
        or product.get("category_name")
        or product.get("category_id"),
        "Uncategorized",
    )


def create_brand(product):
    return clean_text(
        product.get("brand"),
        "Generic",
    )


# ==================================================
# SQL FOR YOUR ACTUAL DATABASE TABLE
# ==================================================
sql = f"""
INSERT INTO `{TABLE_NAME}`
(
    `product_id`,
    `name`,
    `brand`,
    `category`,
    `description`,
    `price`,
    `image`
)
VALUES
(
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s
)
ON DUPLICATE KEY UPDATE
    `name` = VALUES(`name`),
    `brand` = VALUES(`brand`),
    `category` = VALUES(`category`),
    `description` = VALUES(`description`),
    `price` = VALUES(`price`),
    `image` = VALUES(`image`)
"""


# ==================================================
# PREPARE PRODUCT ROWS
# ==================================================
product_rows = []
invalid_products = []

for index, product in enumerate(products, start=1):

    try:
        product_id = clean_text(
            product.get("product_id")
        )

        name = clean_text(
            product.get("name")
            or product.get("product_name")
        )

        if not product_id:
            raise ValueError("Missing product_id")

        if not name:
            raise ValueError("Missing product name")

        brand = create_brand(product)
        category = create_category(product)
        description = create_description(product)

        price = clean_price(
            product.get("base_price")
            if product.get("base_price") is not None
            else product.get("price")
        )

        # Stored relative to Django MEDIA_ROOT.
        #
        # Actual file:
        # media/products/prod_00001.png
        #
        # Database value:
        # products/prod_00001.png
        image_path = (
            f"products/images/{product_id}.{IMAGE_EXTENSION}"
        )

        row = (
            product_id,
            name,
            brand,
            category,
            description,
            price,
            image_path,
        )

        product_rows.append(row)

    except Exception as error:
        invalid_products.append(
            {
                "row_number": index,
                "product_id": product.get(
                    "product_id",
                    "unknown",
                ),
                "error": str(error),
            }
        )


print()
print(f"Valid products: {len(product_rows)}")
print(f"Invalid products: {len(invalid_products)}")


if not product_rows:
    raise ValueError(
        "No valid products were prepared for import."
    )


# ==================================================
# CONNECT TO AMAZON RDS AND IMPORT
# ==================================================
connection = None

try:
    connection = pymysql.connect(
        host=RDS_HOST,
        port=RDS_PORT,
        user=RDS_USER,
        password=RDS_PASSWORD,
        database=RDS_DATABASE,
        charset="utf8mb4",
        autocommit=False,
        connect_timeout=30,
        read_timeout=120,
        write_timeout=120,

        # Explicit AWS RDS certificate.
        ssl_ca=str(SSL_CERTIFICATE.resolve()),
        ssl_verify_cert=True,
        ssl_verify_identity=True,
    )

    print("\nConnected securely to Amazon RDS.")

    # Confirm that the expected table exists.
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_schema = %s
              AND table_name = %s
            """,
            (
                RDS_DATABASE,
                TABLE_NAME,
            ),
        )

        table_exists = cursor.fetchone()[0]

    if table_exists == 0:
        raise ValueError(
            f"The table '{TABLE_NAME}' does not exist "
            f"in database '{RDS_DATABASE}'."
        )

    # Display the actual columns.
    with connection.cursor() as cursor:
        cursor.execute(
            f"SHOW COLUMNS FROM `{TABLE_NAME}`"
        )

        columns = [
            column[0]
            for column in cursor.fetchall()
        ]

    print("Columns found in products_product:")
    print(columns)

    required_columns = {
        "product_id",
        "name",
        "brand",
        "category",
        "description",
        "price",
        "image",
    }

    missing_columns = required_columns - set(columns)

    if missing_columns:
        raise ValueError(
            "The database table is missing these columns: "
            + ", ".join(sorted(missing_columns))
        )

    # Import in batches.
    batch_size = 200
    imported_count = 0

    with connection.cursor() as cursor:

        for start in range(
            0,
            len(product_rows),
            batch_size,
        ):
            batch = product_rows[
                start:start + batch_size
            ]

            cursor.executemany(
                sql,
                batch,
            )

            imported_count += len(batch)

            print(
                f"Processed {imported_count}/"
                f"{len(product_rows)} products"
            )

    connection.commit()

    print()
    print("======================================")
    print("PRODUCT IMPORT COMPLETED SUCCESSFULLY")
    print("======================================")
    print(
        f"Imported or updated: {imported_count}"
    )

    # Verify total records.
    with connection.cursor() as cursor:
        cursor.execute(
            f"SELECT COUNT(*) FROM `{TABLE_NAME}`"
        )

        total_products = cursor.fetchone()[0]

    print(
        f"Total products now in RDS: "
        f"{total_products}"
    )

    # Display five imported products.
    with connection.cursor() as cursor:
        cursor.execute(
            f"""
            SELECT
                product_id,
                name,
                category,
                price,
                image
            FROM `{TABLE_NAME}`
            ORDER BY product_id
            LIMIT 5
            """
        )

        sample_products = cursor.fetchall()

    print("\nFirst five products in RDS:")

    for sample in sample_products:
        print(sample)


except pymysql.MySQLError as error:

    if connection:
        connection.rollback()

    print("\nMySQL import failed:")
    print(error)

    raise


except Exception as error:

    if connection:
        connection.rollback()

    print("\nProduct import failed:")
    print(error)

    raise


finally:

    if connection:
        connection.close()
        print("\nRDS connection closed.")


# ==================================================
# SAVE INVALID PRODUCT ERRORS
# ==================================================
if invalid_products:

    failures_file = Path(
        "product_import_failures.json"
    )

    failures_file.write_text(
        json.dumps(
            invalid_products,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print(
        f"Invalid product details saved to:\n"
        f"{failures_file.resolve()}"
    )

Products found in products.json: 5000

Fields found in the first product:
['base_price', 'category_id', 'creation_date', 'current_stock', 'is_active', 'name', 'price_history', 'product_id', 'subcategory_id']

Valid products: 5000
Invalid products: 0

Connected securely to Amazon RDS.
Columns found in products_product:
['product_id', 'name', 'category', 'brand', 'price', 'description', 'image']
Processed 200/5000 products
Processed 400/5000 products
Processed 600/5000 products
Processed 800/5000 products
Processed 1000/5000 products
Processed 1200/5000 products
Processed 1400/5000 products
Processed 1600/5000 products
Processed 1800/5000 products
Processed 2000/5000 products
Processed 2200/5000 products
Processed 2400/5000 products
Processed 2600/5000 products
Processed 2800/5000 products
Processed 3000/5000 products
Processed 3200/5000 products
Processed 3400/5000 products
Processed 3600/5000 products
Processed 3800/5000 products
Processed 4000/5000 products
Processed 4200/5000 product